# 01 — Dataset Exploration: RVL-CDIP
**Group 1 – Invoices | Step 1**

Esplora `chainyo/rvl-cdip` su tutto il train split disponibile (streaming).  
Grafici: distribuzione classi, dimensioni immagini, brightness, aspect ratio, confronto invoice vs resto, grid visiva.


## 0 — Environment setup

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/NLP_Invoices/outputs'
except ImportError:
    OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output → {OUTPUT_DIR}")


## 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q datasets pillow matplotlib numpy tqdm
print('✅ Done')


## 1 — Imports & configuration

In [ ]:
import os, random
from collections import defaultdict
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from datasets import load_dataset
from tqdm.notebook import tqdm

# ── Global plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  "#0f1117",
    "axes.facecolor":    "#1a1d27",
    "axes.edgecolor":    "#2e3250",
    "axes.labelcolor":   "#c8cde0",
    "xtick.color":       "#8890b0",
    "ytick.color":       "#8890b0",
    "text.color":        "#e0e4f0",
    "grid.color":        "#2e3250",
    "grid.alpha":        0.6,
    "font.family":       "DejaVu Sans",
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "legend.framealpha": 0.3,
    "legend.facecolor":  "#1a1d27",
    "legend.edgecolor":  "#2e3250",
})

ACCENT   = "#e74c3c"   # invoice — rosso
BASE     = "#4e80ee"   # altre classi — blu
MUTED    = "#3a4a7a"   # sfondo barre non evidenziate
GOOD     = "#2ecc71"   # verde per delta positivi
NEUTRAL  = "#95a5a6"

# ── Configuration ────────────────────────────────────────────────────────────────────
N_PER_CLASS   = 200      # esempi per classe (16 × 200 = 3200 totali)
INVOICE_LABEL = 11
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

LABEL_NAMES = {
    0:"letter", 1:"form", 2:"email", 3:"handwritten", 4:"advertisement",
    5:"scientific report", 6:"scientific publication", 7:"specification",
    8:"file folder", 9:"news article", 10:"budget", 11:"invoice",
    12:"presentation", 13:"questionnaire", 14:"resume", 15:"memo"
}
ALL_LABELS  = list(LABEL_NAMES.keys())
CLASS_NAMES = [LABEL_NAMES[i] for i in ALL_LABELS]
NUM_CLASSES = len(CLASS_NAMES)

COLOURS = [ACCENT if l == INVOICE_LABEL else BASE for l in ALL_LABELS]
print(f"Config ready — {N_PER_CLASS}/class × {NUM_CLASSES} = {N_PER_CLASS*NUM_CLASSES} total examples")


## 2 — Stream dataset (balanced sampling)

In [ ]:
def to_pil(image_field):
    if isinstance(image_field, Image.Image):
        return image_field.convert("RGB")
    elif isinstance(image_field, dict):
        return Image.open(BytesIO(image_field["bytes"])).convert("RGB")
    else:
        return Image.fromarray(image_field).convert("RGB")

print(f"Streaming {N_PER_CLASS}/class × {NUM_CLASSES} classes from train split...")

stream    = load_dataset("chainyo/rvl-cdip", split="train", streaming=True
            ).shuffle(seed=RANDOM_SEED, buffer_size=20_000)
balanced  = defaultdict(list)
completed = set()

for example in stream:
    label = int(example["label"])
    if label in completed:
        continue
    balanced[label].append(example)
    if len(balanced[label]) >= N_PER_CLASS:
        completed.add(label)
        print(f"  ✓ {LABEL_NAMES[label]:<26} ({N_PER_CLASS})")
    if len(completed) == NUM_CLASSES:
        break

# Flatten
all_examples = [ex for exs in balanced.values() for ex in exs]
random.shuffle(all_examples)

all_images = [to_pil(ex["image"]) for ex in tqdm(all_examples, desc="Converting images")]
all_labels = [int(ex["label"]) for ex in all_examples]

invoice_images = [to_pil(ex["image"]) for ex in balanced[INVOICE_LABEL]]
print(f"\n✅ Total: {len(all_images)} images  ({len(invoice_images)} invoice)")


## 3 — Class distribution

In [ ]:
counts  = [len(balanced[l]) for l in ALL_LABELS]
colours = [ACCENT if l == INVOICE_LABEL else BASE for l in ALL_LABELS]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(CLASS_NAMES, counts, color=colours, width=0.7,
              edgecolor="#0f1117", linewidth=0.8)

# Value labels above bars
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            str(c), ha="center", va="bottom", fontsize=8, color="#8890b0")

# Mean reference line
mean_c = np.mean(counts)
ax.axhline(mean_c, color=NEUTRAL, linestyle="--", linewidth=1, alpha=0.7)
ax.text(NUM_CLASSES - 0.4, mean_c + 1, f"media {mean_c:.0f}",
        color=NEUTRAL, fontsize=8, ha="right")

ax.set_title("RVL-CDIP — class distribution in the train set", pad=14)
ax.set_ylabel("Sampled examples")
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=38, ha="right", fontsize=9)
ax.set_ylim(0, max(counts) * 1.15)
ax.spines[["top","right","left","bottom"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5)
ax.set_axisbelow(True)

legend_elems = [
    mpatches.Patch(facecolor=ACCENT, label="invoice"),
    mpatches.Patch(facecolor=BASE,   label="other classes"),
]
ax.legend(handles=legend_elems, fontsize=9, loc="upper right")

plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "01_class_distribution.png")
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show(); print(f"Saved → {path}")


## 4 — Image dimensions & aspect ratio

In [ ]:
widths  = np.array([img.width  for img in all_images])
heights = np.array([img.height for img in all_images])
aspects = widths / heights

class_w  = {l: np.mean([img.width  for img,lb in zip(all_images,all_labels) if lb==l]) for l in ALL_LABELS}
class_h  = {l: np.mean([img.height for img,lb in zip(all_images,all_labels) if lb==l]) for l in ALL_LABELS}
class_ar = {l: np.mean([img.width/img.height for img,lb in zip(all_images,all_labels) if lb==l]) for l in ALL_LABELS}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, data_dict, xlabel, title in [
    (axes[0], class_w,  "pixel", "Mean width (px)"),
    (axes[1], class_h,  "pixel", "Mean height (px)"),
    (axes[2], class_ar, "w / h", "Mean aspect ratio"),
]:
    vals = [data_dict[l] for l in ALL_LABELS]
    cols = [ACCENT if l == INVOICE_LABEL else BASE for l in ALL_LABELS]
    bars = ax.barh(CLASS_NAMES, vals, color=cols, height=0.7,
                   edgecolor="#0f1117", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(v + max(vals)*0.01, bar.get_y() + bar.get_height()/2,
                f"{v:.0f}" if xlabel == "pixel" else f"{v:.2f}",
                va="center", fontsize=7.5, color="#8890b0")
    ax.set_xlabel(xlabel)
    ax.set_title(title, pad=10)
    ax.spines[["top","right","left","bottom"]].set_visible(False)
    ax.xaxis.grid(True, linewidth=0.5)
    ax.set_axisbelow(True)
    if xlabel == "w / h":
        ax.axvline(1.0, color=NEUTRAL, linestyle="--", linewidth=1, alpha=0.7)

plt.suptitle("Image dimension statistics per class", fontsize=14, y=1.01)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "01_image_sizes.png")
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show(); print(f"Saved → {path}")

print(f"\nStats globali:")
print(f"  Larghezza  — media: {widths.mean():.0f}px  std: {widths.std():.0f}px  range: {widths.min()}–{widths.max()}")
print(f"  Altezza    — media: {heights.mean():.0f}px  std: {heights.std():.0f}px  range: {heights.min()}–{heights.max()}")
print(f"  Aspect     — media: {aspects.mean():.2f}  std: {aspects.std():.2f}")


## 5 — Brightness per class

In [ ]:
def mean_brightness(img):
    return np.array(img.convert("L")).mean()

print("Computing brightness values...")
brightnesses = [mean_brightness(img) for img in tqdm(all_images, desc="Computing brightness")]

class_bright = defaultdict(list)
for b, l in zip(brightnesses, all_labels):
    class_bright[l].append(b)

# ── Violinplot + strip ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))

positions = list(range(1, NUM_CLASSES + 1))
parts = ax.violinplot(
    [class_bright[l] for l in ALL_LABELS],
    positions=positions,
    showmedians=True,
    showextrema=False,
    widths=0.7,
)
for i, (pc, l) in enumerate(zip(parts["bodies"], ALL_LABELS)):
    pc.set_facecolor(ACCENT if l == INVOICE_LABEL else BASE)
    pc.set_alpha(0.75)
parts["cmedians"].set_color("#ffffff")
parts["cmedians"].set_linewidth(1.5)

# Per-class mean as scatter point
means = [np.mean(class_bright[l]) for l in ALL_LABELS]
ax.scatter(positions, means, color="#ffffff", s=18, zorder=5, alpha=0.9)

ax.set_xticks(positions)
ax.set_xticklabels(CLASS_NAMES, rotation=38, ha="right", fontsize=9)
ax.set_ylabel("Mean brightness (0=black, 255=white)")
ax.set_title("Brightness distribution per class — violin plot", pad=14)
ax.spines[["top","right","left","bottom"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5)
ax.set_axisbelow(True)

legend_elems = [
    mpatches.Patch(facecolor=ACCENT, alpha=0.75, label="invoice"),
    mpatches.Patch(facecolor=BASE,   alpha=0.75, label="other classes"),
]
ax.legend(handles=legend_elems, fontsize=9)

plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "01_brightness.png")
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show(); print(f"Saved → {path}")


## 6 — Invoice vs. rest: comparative analysis

In [ ]:
inv_b   = [b for b, l in zip(brightnesses, all_labels) if l == INVOICE_LABEL]
other_b = [b for b, l in zip(brightnesses, all_labels) if l != INVOICE_LABEL]

inv_w   = [img.width  for img, l in zip(all_images, all_labels) if l == INVOICE_LABEL]
inv_h   = [img.height for img, l in zip(all_images, all_labels) if l == INVOICE_LABEL]
inv_ar  = [w/h for w, h in zip(inv_w, inv_h)]
oth_w   = [img.width  for img, l in zip(all_images, all_labels) if l != INVOICE_LABEL]
oth_h   = [img.height for img, l in zip(all_images, all_labels) if l != INVOICE_LABEL]
oth_ar  = [w/h for w, h in zip(oth_w, oth_h)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── 1: Donut chart ──────────────────────────────────────────────────────────
ax = axes[0, 0]
n_inv   = len(balanced[INVOICE_LABEL])
n_other = sum(len(balanced[l]) for l in ALL_LABELS if l != INVOICE_LABEL)
wedge_colors = [ACCENT, MUTED]
wedges, texts, autotexts = ax.pie(
    [n_inv, n_other],
    labels=["invoice", "other classes"],
    colors=wedge_colors,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops=dict(edgecolor="#0f1117", linewidth=2, width=0.55),
    textprops=dict(color="#e0e4f0", fontsize=10),
)
for at in autotexts:
    at.set_color("#ffffff"); at.set_fontsize(11); at.set_fontweight("bold")
ax.set_title("Invoice vs. other classes (balanced sample)", pad=10)

# ── 2: Brightness histogram ───────────────────────────────────────
ax = axes[0, 1]
bins = np.linspace(0, 255, 40)
ax.hist(other_b, bins=bins, alpha=0.55, color=BASE,   label=f"other ({len(other_b)} imgs)",
        edgecolor="#0f1117", linewidth=0.3)
ax.hist(inv_b,   bins=bins, alpha=0.75, color=ACCENT, label=f"invoice ({len(inv_b)} imgs)",
        edgecolor="#0f1117", linewidth=0.3)
ax.axvline(np.mean(inv_b),   color=ACCENT, linestyle="--", linewidth=1.5, alpha=0.9)
ax.axvline(np.mean(other_b), color=BASE,   linestyle="--", linewidth=1.5, alpha=0.9)
ax.set_xlabel("Brightness media"); ax.set_ylabel("Conteggio")
ax.set_title("Brightness distribution: invoice vs. rest")
ax.legend(fontsize=9)
ax.spines[["top","right","left","bottom"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5); ax.set_axisbelow(True)

# ── 3: Aspect ratio histogram ───────────────────────────────────────────────
ax = axes[1, 0]
bins_ar = np.linspace(0, 3, 40)
ax.hist(oth_ar, bins=bins_ar, alpha=0.55, color=BASE,   label="altre",   edgecolor="#0f1117", linewidth=0.3)
ax.hist(inv_ar, bins=bins_ar, alpha=0.75, color=ACCENT, label="invoice", edgecolor="#0f1117", linewidth=0.3)
ax.axvline(1.0, color=NEUTRAL, linestyle="--", linewidth=1, alpha=0.7, label="1:1 (square)")
ax.axvline(np.mean(inv_ar),   color=ACCENT, linestyle=":",  linewidth=1.5)
ax.axvline(np.mean(oth_ar),   color=BASE,   linestyle=":",  linewidth=1.5)
ax.set_xlabel("Aspect ratio (w/h)"); ax.set_ylabel("Conteggio")
ax.set_title("Aspect ratio: invoice vs. rest")
ax.legend(fontsize=9)
ax.spines[["top","right","left","bottom"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5); ax.set_axisbelow(True)

# ── 4: Scatter width vs. height ──────────────────────────────────────────────
ax = axes[1, 1]
oth_idx = [i for i, l in enumerate(all_labels) if l != INVOICE_LABEL]
inv_idx_list = [i for i, l in enumerate(all_labels) if l == INVOICE_LABEL]
sample_oth = random.sample(oth_idx, min(400, len(oth_idx)))
ax.scatter([widths[i] for i in sample_oth], [heights[i] for i in sample_oth],
           color=BASE,   alpha=0.3, s=12, label="altre (campione 400)", linewidths=0)
ax.scatter([widths[i] for i in inv_idx_list], [heights[i] for i in inv_idx_list],
           color=ACCENT, alpha=0.7, s=20, label="invoice", linewidths=0)
ax.set_xlabel("Larghezza (px)"); ax.set_ylabel("Altezza (px)")
ax.set_title("Scatter: width vs. height")
ax.legend(fontsize=9)
ax.spines[["top","right","left","bottom"]].set_visible(False)
ax.grid(True, linewidth=0.5); ax.set_axisbelow(True)

plt.suptitle("Invoice vs. rest — comparative analysis", fontsize=14, y=1.01)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "01_invoice_vs_rest.png")
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show(); print(f"Saved → {path}")

print(f"\nInvoice brightness  — media: {np.mean(inv_b):.1f}  std: {np.std(inv_b):.1f}")
print(f"Altre    brightness  — media: {np.mean(other_b):.1f}  std: {np.std(other_b):.1f}")
print(f"Invoice aspect ratio — media: {np.mean(inv_ar):.2f}  std: {np.std(inv_ar):.2f}")
print(f"Altre   aspect ratio — media: {np.mean(oth_ar):.2f}  std: {np.std(oth_ar):.2f}")


## 7 — Per-class feature correlation heatmap

In [ ]:
# Per-class features: mean brightness, mean width, mean height, aspect ratio
feature_matrix = np.array([
    [np.mean(class_bright[l]),
     class_w[l],
     class_h[l],
     class_ar[l]]
    for l in ALL_LABELS
])

# Normalise each feature to [0, 1]
feature_norm = (feature_matrix - feature_matrix.min(0)) / (feature_matrix.max(0) - feature_matrix.min(0) + 1e-8)
feature_names = ["Brightness", "Larghezza", "Altezza", "Aspect ratio"]

fig, ax = plt.subplots(figsize=(8, 10))
cmap = LinearSegmentedColormap.from_list("custom", ["#1a1d27", "#4e80ee", "#e74c3c"])
im = ax.imshow(feature_norm, aspect="auto", cmap=cmap, vmin=0, vmax=1)

ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, fontsize=10)
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(CLASS_NAMES, fontsize=9)

# Cell value annotations
for i in range(NUM_CLASSES):
    for j in range(len(feature_names)):
        ax.text(j, i, f"{feature_norm[i,j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="white" if feature_norm[i,j] < 0.7 else "#0f1117")

# Evidenzia riga invoice
inv_row = ALL_LABELS.index(INVOICE_LABEL)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.add_patch(plt.Rectangle((-0.5, inv_row - 0.5), len(feature_names), 1,
                             fill=False, edgecolor=ACCENT, linewidth=2.5))

plt.colorbar(im, ax=ax, label="Valore normalizzato (0–1)", shrink=0.6, pad=0.02)
ax.set_title("Heatmap feature per classe (normalizzate 0–1)
bordo rosso = invoice", pad=14)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "01_feature_heatmap.png")
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show(); print(f"Saved → {path}")


## 8 — Invoice visual grid

In [ ]:
n_show = min(16, len(invoice_images))
chosen = random.sample(invoice_images, n_show)
cols, rows = 4, 4

fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.8))
fig.patch.set_facecolor("#0f1117")
axes = axes.flatten()

for idx, img in enumerate(chosen):
    axes[idx].imshow(img)
    axes[idx].set_title(f"Invoice #{idx+1}", fontsize=9, color="#c8cde0", pad=4)
    axes[idx].axis("off")
    for spine in axes[idx].spines.values():
        spine.set_edgecolor(ACCENT); spine.set_linewidth(1.5); spine.set_visible(True)

for idx in range(n_show, len(axes)):
    axes[idx].axis("off")

plt.suptitle(f"Campione visivo: {n_show} invoice da RVL-CDIP", fontsize=13,
             color="#e0e4f0", y=1.01)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "01_invoice_grid.png")
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show(); print(f"Saved → {path}")


## 9 — All-classes visual grid (1 sample per class)

In [ ]:
cols, rows = 4, 4
fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.8))
fig.patch.set_facecolor("#0f1117")
axes = axes.flatten()

for idx, lbl in enumerate(ALL_LABELS):
    img = to_pil(balanced[lbl][0]["image"])
    axes[idx].imshow(img)
    is_inv = lbl == INVOICE_LABEL
    axes[idx].set_title(LABEL_NAMES[lbl], fontsize=9,
                         color=ACCENT if is_inv else "#c8cde0", pad=4,
                         fontweight="bold" if is_inv else "normal")
    axes[idx].axis("off")
    color = ACCENT if is_inv else "#2e3250"
    lw    = 2.0   if is_inv else 1.0
    for spine in axes[idx].spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(lw); spine.set_visible(True)

plt.suptitle("RVL-CDIP — un esempio per classe  (bordo rosso = invoice)",
             fontsize=13, color="#e0e4f0", y=1.01)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "01_all_classes_grid.png")
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show(); print(f"Saved → {path}")


## 10 — Save summary

In [ ]:
inv_bright_mean = np.mean(class_bright[INVOICE_LABEL])
inv_w_mean      = class_w[INVOICE_LABEL]
inv_h_mean      = class_h[INVOICE_LABEL]
inv_ar_mean     = class_ar[INVOICE_LABEL]

summary_path = os.path.join(OUTPUT_DIR, "01_dataset_summary.txt")
with open(summary_path, "w") as f:
    f.write("RVL-CDIP (chainyo) — Exploration Summary\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Campione bilanciato : {len(all_images)} esempi ({N_PER_CLASS}/classe × {NUM_CLASSES} classi)\n")
    f.write(f"Invoice nel campione: {len(invoice_images)}\n\n")
    f.write("Statistiche globali:\n")
    f.write(f"  Larghezza media/std : {widths.mean():.0f} / {widths.std():.0f} px\n")
    f.write(f"  Altezza   media/std : {heights.mean():.0f} / {heights.std():.0f} px\n")
    f.write(f"  Aspect ratio medio  : {aspects.mean():.2f}\n\n")
    f.write("Statistiche invoice:\n")
    f.write(f"  Larghezza media     : {inv_w_mean:.0f} px\n")
    f.write(f"  Altezza   media     : {inv_h_mean:.0f} px\n")
    f.write(f"  Aspect ratio medio  : {inv_ar_mean:.2f}\n")
    f.write(f"  Brightness media    : {inv_bright_mean:.1f}\n\n")
    f.write("Conteggio per classe:\n")
    for l in ALL_LABELS:
        tag = " ← invoice" if l == INVOICE_LABEL else ""
        f.write(f"  {LABEL_NAMES[l]:<26} {len(balanced[l])}{tag}\n")

print(f"Summary salvato → {summary_path}")
print("\n✅ Step 1 completo. Output salvati in:", OUTPUT_DIR)
print("   Grafici generati:")
for fname in ["01_class_distribution.png", "01_image_sizes.png", "01_brightness.png",
              "01_invoice_vs_rest.png", "01_feature_heatmap.png",
              "01_invoice_grid.png", "01_all_classes_grid.png"]:
    print(f"   • {fname}")
